Vogliamo costruire il dataset dei prodotti Amazon nel campo Electronics, intersecando:
Amazon Reviews 2023 che contiene metadati + rating/recensioni
Amazon question/answer storico McAuley

Questo notebook costruisce il dataset Electronics intersecando:
- QA storico McAuley
- metadati Amazon Reviews 2023
- rating/recensioni Amazon Reviews 2023

Output finale:
- items.parquet
- qa.parquet
- ratings.parquet
- summary.json

In [1]:
# Se mancano pacchetti, esegui questa cella una sola volta.
%pip install -q "datasets==3.6.0" pandas pyarrow requests tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path #per gestire meglio percorsi di file e cartelle
import ast #servirà più avanti per leggere file del QA storico
import gzip #leggere e scrivere file compressi gz
import json #lavorare con json
from typing import Iterable, Optional #tipi
import pandas as pd #pandas per creare dataframe
import pyarrow as pa #per definire schemi tabellari
import pyarrow.parquet as pq    #per leggere metadata dei file .parquet
import requests #per scaricare da URL
from tqdm.auto import tqdm #barra di progresso
import datasets #pacchetto datasets
from datasets import load_dataset #per caricare dataset

datasets.logging.set_verbosity_error() #riduce i messaggi stampati da datasets

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#configurazione
CATEGORY_2023="Electronics" #modifica qua per cambiare categoria

QA_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/qa_Electronics.json.gz"
#per versione risposte multiple https://mcauleylab.ucsd.edu/public_datasets/data/amazon/qa/icdm/QA_Electronics.json.gz

In [4]:
PROJECT_ROOT=Path("..").resolve() if Path("..").joinpath("data").exists() else Path(".").resolve() #sceglie root
RAW_DIR=PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR=PROJECT_ROOT / "data" / "processed" / "electronics_intersection"

QA_RAW_PATH=RAW_DIR / "qa" / "qa_Electronics.json.gz"
QA_STRICT_PATH=RAW_DIR / "qa" / "qa_Electronics.strict.jsonl.gz"

ITEMS_OUT=PROCESSED_DIR / "items.parquet" #percorso finale dei prodotti
QA_OUT=PROCESSED_DIR / "qa.parquet" #percorso file domande/risposte
RATINGS_OUT=PROCESSED_DIR / "ratings.parquet" #percorso rating

RAW_DIR.mkdir(parents=True, exist_ok=True) #crea data/raw
QA_RAW_PATH.parent.mkdir(parents=True, exist_ok=True) #crea padre data/raw/qa
PROCESSED_DIR.mkdir(parents=True, exist_ok=True) #crea data/processed/electronics_intersection

print("Project root:", PROJECT_ROOT)
print("Output:", PROCESSED_DIR)

Project root: C:\Users\arsid\Desktop\Tesi-ecommerce
Output: C:\Users\arsid\Desktop\Tesi-ecommerce\data\processed\electronics_intersection


In [5]:
def download_file(url:str,output_path:Path, force:bool=False) ->Path: #scarica da url in output_path
    if output_path.exists() and not force: #non riscarica
        print(f"Già presente: {output_path}")
        return output_path
    output_path.parent.mkdir(parents=True, exist_ok=True) #se non esiste cartella di destinazione la crea

    with requests.get(url, stream=True,timeout=60)as r:
        r.raise_for_status()
        total =int(r.headers.get("content-length", 0)) #dimensione
        with open(output_path, "wb") as f, tqdm( #apre file in scrittura binaria, perchè scarichi .gz e apre barra tqdm
            total=total,
            unit="B", #in byte
            unit_scale=True,
            desc=output_path.name,
        ) as pbar:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    return output_path

download_file(QA_URL, QA_RAW_PATH)

Già presente: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.json.gz


WindowsPath('C:/Users/arsid/Desktop/Tesi-ecommerce/data/raw/qa/qa_Electronics.json.gz')

In [6]:
#Convertiamo QA storico da JSONL gzip a JSON e aggiungiamo parent_asin per
#agganciarlo al dataset amazon 2023
#per fare questo l'asin storico lo usiamo come parent_asin
def iter_historical_qa(path: Path)->Iterable[dict]: #legge qa storico, restituisce dizionari python
    with gzip.open(path,"rt",encoding="utf-8", errors="replace") as f: #apre il file .gz
        for line in f:
            line =line.strip() #rimuove spazi e newline
            if line:
                yield ast.literal_eval(line) #converte riga da testo a dizionario python

qa_asins=set()
qa_rows=0


with gzip.open(QA_STRICT_PATH, "wt", encoding="utf-8") as out: #apre il file in scrittura, .jsonl.gz
    for qa_id,row in enumerate(tqdm(iter_historical_qa(QA_RAW_PATH), desc="Conversione QA")): #scorre tutti i record del qa storico
        asin=row.get("asin") #prende valore dal campo asin
        row["qa_id"] = qa_id #aggiunge id numerico alla riga qa
        row["parent_asin"] = asin #crea il campo parent_asin e ci copia l'asin

        if asin: #se asin non è vuoto, aggiunge l'asin all'insieme dei prodotti presenti nel qa
            qa_asins.add(asin)

        out.write(json.dumps(row, ensure_ascii=False) + "\n") #scrive la riga nel nuovo file jsonl
        qa_rows += 1

print("QA rows:", qa_rows)
print("Prodotti unici nel QA:", len(qa_asins))
print("File strict:", QA_STRICT_PATH)

Conversione QA: 314263it [00:14, 22174.26it/s]

QA rows: 314263
Prodotti unici nel QA: 39371
File strict: C:\Users\arsid\Desktop\Tesi-ecommerce\data\raw\qa\qa_Electronics.strict.jsonl.gz


In [7]:
#carichiamo metadati amazon reviews 2023 da Hugging Face (piattaforma dove è pubblicato il dataset)
meta_ds = load_dataset( #risultato salvato in meta_ds
    "McAuley-Lab/Amazon-Reviews-2023", #nome del dataset su HuggingFace
    f"raw_meta_{CATEGORY_2023}", #raw_meta_electronics per ora
    split="full", 
    trust_remote_code=True,
)
print(meta_ds)
print("Numero metadati:", len(meta_ds))

Dataset({
    features: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
    num_rows: 1610012
})
Numero metadati: 1610012


In [8]:
def image_url(images)->Optional[str]: #prova a estrarre immagine da campo images
    if isinstance(images,dict):
        for key in ("hi_res","large","thumb"): #prova alcune chiavi
            values=images.get(key) or []
            for value in values:
                if value:
                    return value
    if isinstance(images, list):
        for img in images:
            if isinstance(img, dict):
                for key in ("large_image_url","medium_image_url","small_image_url"):
                    value=img.get(key)
                    if value:
                        return value
    return None

def tojson_string(value)->str: #converte valore in stringa json
    if value is None:
        return "null"
    return json.dumps(value, ensure_ascii=False)

def tofloat_or_none(value): #converte valore in numero decimale
    try:
        if value in (None,"","None"):
            return None
        return float(value)
    except Exception:
        return None

def get_category(row: dict) ->Optional[str]:
    main_category=row.get("main_category")
    if main_category not in (None,"", "None"):
        return main_category

    categories=row.get("categories")
    if isinstance(categories, list) and len(categories)>0:
        return categories[0]

    return CATEGORY_2023

def toint_or_none(value): #converte in intero
    try:
        if value in (None,"", "None"):
            return None
        return int(value)
    except Exception:
        return None

ITEM_SCHEMA = pa.schema([ #schema del file items.parquet
    ("parent_asin", pa.string()),
    ("title", pa.string()),
    ("category", pa.string()),
    ("avg_rating", pa.float64()),
    ("num_ratings", pa.int64()),
    ("price", pa.float64()),
    ("store", pa.string()),
    ("features_json", pa.string()),
    ("description_json", pa.string()),
    ("categories_json", pa.string()),
    ("details_json", pa.string()),
    ("image_url", pa.string()),
])

#scrive righe in un file parquet, rows=listadiz, path=dove salvare, schema=schema PyArrow, writer=oggetto che gestisce la scrittura Parquet
def writerows_in_parquet(rows:list[dict], path:Path, schema: pa.schema, writer:Optional[pq.ParquetWriter]):
    if not rows:
        return writer

    df=pd.DataFrame(rows) #converte lista di dizionari in un dataframe pandas
    table=pa.Table.from_pandas(df, schema=schema, preserve_index=False) #converte dataframe in tabella pyarrow

    if writer is None:
        path.parent.mkdir(parents=True, exist_ok=True) #crea cartella di output
        writer=pq.ParquetWriter(path, schema=schema, compression="zstd") #crea un writer Parquet,compresso
    writer.write_table(table)
    return writer

#dobbiamo filtrare solo elementi presenti anche nel QA storico
if ITEMS_OUT.exists(): #se items.parquet esiste già lo elimina
    ITEMS_OUT.unlink()

writer=None
buffer=[]
matched_items=0

for row in tqdm(meta_ds, desc="Filtro metadati"): #scorre tutti i prodotti nei metadati amazon 2023 electronics
    parent_asin=row.get("parent_asin")
    if parent_asin not in qa_asins:
        continue
    buffer.append({ #se prodotto è presente nel qa, lo aggiunge al buffer
        "parent_asin": parent_asin,
        "title": row.get("title"),
        "category":get_category(row),
        "avg_rating": tofloat_or_none(row.get("average_rating")),
        "num_ratings": toint_or_none(row.get("rating_number")),
        "price": tofloat_or_none(row.get("price")),
        "store": row.get("store"),
        "features_json": tojson_string(row.get("features")),
        "description_json": tojson_string(row.get("description")),
        "categories_json": tojson_string(row.get("categories")),
        "details_json": tojson_string(row.get("details")),
        "image_url": image_url(row.get("images")),
    })
    matched_items+=1

    if len(buffer) >= 50_000:
        writer=writerows_in_parquet(buffer, ITEMS_OUT, ITEM_SCHEMA, writer)
        buffer=[]
writer = writerows_in_parquet(buffer, ITEMS_OUT, ITEM_SCHEMA, writer) #nel caso avesse meno di 50000 el.    
if writer is not None:
    writer.close()

items_df=pd.read_parquet(ITEMS_OUT)
items_id=set(items_df["parent_asin"])

print("Prodotti QA ∩ metadata 2023:", len(items_id))
display(items_df.head())


Filtro metadati: 100%|██████████| 1610012/1610012 [02:38<00:00, 10158.73it/s]


Prodotti QA ∩ metadata 2023: 22975


,parent_asin,title,category,avg_rating,num_ratings,price,store,features_json,description_json,categories_json,details_json,image_url
0,B00CFGZAT8,Dock Audio Extender Adapter Converter Cable fo...,Portable Audio & Accessories,3.6,45,NaN,iteck,[],[],"[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Brand Name\"": \""iTeck\"", \""Item Weight\"": ...",https://m.media-amazon.com/images/I/41uTuqlHDS...
1,B00BGFTSGU,GGS Swivi HD DSLR LCD Universal Foldable Viewf...,Camera & Photo,3.7,52,149.00,Swivi,[],[],"[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Product Dimensions\"": \""3.8 x 4.3 x 6.9 in...",https://m.media-amazon.com/images/I/41BUv5PjMD...
2,B00AZOIG9I,BM Premium 2-Pack of NP-85 Batteries and Charg...,Camera & Photo,4.7,345,22.99,BM Premium,"[""NP85 Li-ion Battery for FujiFilm FinePix SL2...","[""Bring your digital camera back to life with ...","[""Electronics"", ""Camera & Photo"", ""Accessories...","""{\""Package Dimensions\"": \""6.46 x 3.94 x 1.85...",https://m.media-amazon.com/images/I/71fbU-87Tx...
3,B00F361KLO,ZAGG InvisibleShield HD – EZ Apply Film Screen...,Computers,4.1,158,NaN,InvisibleShield,"[""100% Clear: Independent light transmission t...","[""Exceptionally clear, unbelievably thin, and ...","[""Electronics"", ""Computers & Accessories"", ""Ta...","""{\""Standing screen display size\"": \""9.7 Inch...",https://m.media-amazon.com/images/I/71XY4kKwcX...
4,B0029L7N76,Mobile Edge ECO Laptop Messenger (Eco-Friendly...,Computers,3.5,12,NaN,Mobile Edge,"[""Fits 17.3-Inch laptops. Laptop compartment d...","[""The 17.3-Inch ECO Messenger is the latest ad...","[""Electronics"", ""Computers & Accessories"", ""La...","""{\""Product Dimensions\"": \""4.25 x 18 x 13.5 i...",https://m.media-amazon.com/images/I/91JdX5IqgA...


In [9]:
# ora filtriamo QA e teniamo solo domande/risposte dei prodotti presenti anche nei metadati 2023
QA_SCHEMA=pa.schema([ #schema del file pa.parquet
    ("qa_id", pa.int64()),
    ("parent_asin", pa.string()),
    ("question", pa.string()),
    ("answer", pa.string()),
    ("questionType", pa.string()),
    ("answerType", pa.string()),
    ("answerTime", pa.string()),
    ("unixTime", pa.int64()),
])

def iter_jsonl_gz(path: Path) -> Iterable[dict]: #legge file .jsonl.gz
    with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f: #apre file compresso in modalità testo
        for line in f:
            line=line.strip() #toglie spazi e newline
            if line:
                yield json.loads(line) #converte la riga json in dizionario python

if QA_OUT.exists(): #se qa.parquet esiste già lo elimina
    QA_OUT.unlink()

writer= None
buffer = []
matched_qa = 0

for row in tqdm(iter_jsonl_gz(QA_STRICT_PATH), desc="Filtro QA"): #con tqdm abbiamo la riga di progresso, scorriamo il qa convertito
    parent_asin=row.get("parent_asin")

    if parent_asin not in items_id:
        continue

    buffer.append({ #aggiunge una riga al buffer
        "qa_id": int(row.get("qa_id")),
        "parent_asin": parent_asin,
        "question": row.get("question"),
        "answer": row.get("answer"),
        "questionType": row.get("questionType"),
        "answerType": row.get("answerType"),
        "answerTime": row.get("answerTime"),
        "unixTime": toint_or_none(row.get("unixTime")),
    })
    matched_qa+=1

    if len(buffer)>= 100_000: #se buffer ha almeno 100000 righe
        writer = writerows_in_parquet(buffer, QA_OUT, QA_SCHEMA, writer)
        buffer=[]
    
writer = writerows_in_parquet(buffer, QA_OUT, QA_SCHEMA, writer) 
if writer is not None: #se è stato scritto qualcosa, chiudi il writer
    writer.close()

qa_df = pd.read_parquet(QA_OUT) #legge qa filtrato
print("QA filtrate:", len(qa_df))
display(qa_df.head())


Filtro QA: 314263it [00:01, 189820.95it/s]


QA filtrate: 180986


,qa_id,parent_asin,question,answer,questionType,answerType,answerTime,unixTime
0,0,0594033926,Is this cover the one that fits the old nook c...,Yes this fits both the nook color and the same...,yes/no,Y,"Dec 27, 2013",1.388131e+09
1,1,0594033926,Does it fit Nook GlowLight?,No. The nook color or color tablet,yes/no,N,"Jan 5, 2015",1.420445e+09
2,2,0594033926,Would it fit Nook 1st Edition? 4.9in x 7.7in ?,I don't think so. The nook color is 5 x 8 so n...,open-ended,NaN,2 days ago,NaN
3,3,0594033926,Will this fit a Nook Color that's 5 x 8?,yes,yes/no,Y,17 days ago,NaN
4,4,0594033926,will this fit the Samsung Galaxy Tab 4 Nook 10.1,"No, the tab is smaller than the 'color'",yes/no,N,"Feb 10, 2015",1.423555e+09


In [ ]:
#scansioniamo reviews 2023 electronics e salviamo solo interazioni dei prodotti che abbiamo già
MAX_REVIEWS_SCAN = None
SAVE_TEXT_OFREVIEW = False
FORCE_REBUILD_RATINGS = False #per non ricaricare sempre tutto, ci mette circa 45min

if RATINGS_OUT.exists() and not FORCE_REBUILD_RATINGS:
    n_rows = pq.ParquetFile(RATINGS_OUT).metadata.num_rows
    print(f"ratings.parquet già presente: {RATINGS_OUT}")
else:
    review_stream = load_dataset( #carica dataset delle reviews in streaming
        "McAuley-Lab/Amazon-Reviews-2023", #dataset hugging face
        f"raw_review_{CATEGORY_2023}", #electronics per ora
        split="full", 
        streaming=True, #letto progressivamente senza scaricare tutto in RAM
        trust_remote_code=True,
    )
    rating_fields = [ #lista di colonne per file di rating
        ("user_id", pa.string()),
        ("parent_asin", pa.string()),
        ("asin", pa.string()),
        ("rating", pa.float64()),
        ("timestamp", pa.int64()),
        ("verified_purchase", pa.bool_()),
    ]
    if SAVE_TEXT_OFREVIEW: #se true
        rating_fields.extend([
            ("review_title", pa.string()),
            ("review_text", pa.string()),
        ])
    RATING_SCHEMA = pa.schema(rating_fields) #crea schema pyarrow per ratings.parquet

    if RATINGS_OUT.exists(): #se file esiste già, lo elimina
        RATINGS_OUT.unlink()

    writer = None
    buffer = []
    matched_ratings = 0
    scanned =0

    for row in tqdm(review_stream, desc="Filtro rating"): #scorri recensioni una alla volta in streaming
        if MAX_REVIEWS_SCAN is not None and scanned >= MAX_REVIEWS_SCAN: #se hai raggiunto limite, se impostato
            break
        scanned+=1

        parent_asin=row.get("parent_asin")#prende parent asin

        if parent_asin not in items_id: #se non è nella intersezione finale, lo salta
            continue

        out_row={
            "user_id": row.get("user_id"),
            "parent_asin": parent_asin,
            "asin": row.get("asin"),
            "rating": tofloat_or_none(row.get("rating")),
            "timestamp": toint_or_none(row.get("timestamp")),
            "verified_purchase": row.get("verified_purchase"),
        }
        if SAVE_TEXT_OFREVIEW:
            out_row["review_title"] = row.get("title")
            out_row["review_text"] = row.get("text")

        buffer.append(out_row)
        matched_ratings +=1


        if len(buffer) >= 200_000:
            writer=writerows_in_parquet(buffer, RATINGS_OUT, RATING_SCHEMA, writer)
            buffer=[]

    writer=writerows_in_parquet(buffer, RATINGS_OUT, RATING_SCHEMA, writer)
    if writer is not None:
        writer.close()

    print("Review scansionate:", scanned)
    print("Rating salvati:", matched_ratings)
    print("Output rating:", RATINGS_OUT)

NameError: name 'RATINGS_OUT' is not defined

In [11]:
#allineamento items e qa ai soli prodotti che hanno rating, era uscito fuori che un solo prodotto non aveva rating, quindi lo escludiamo
ratings_df=pd.read_parquet(RATINGS_OUT,columns=["parent_asin"]) #carica solo colonna parent_asin

items_df=items_df[
    items_df["parent_asin"].isin(set(ratings_df["parent_asin"]))
].copy() #tiene solo i prodotti il cui parentasin compare anche dentro ratings_df

items_df.to_parquet(ITEMS_OUT, index=False)  #riscrive il file
items_id = set(items_df["parent_asin"]) #aggiorna variabile items_id 

print("Prodotti finali con metadata + QA + rating:", len(items_df))

valid_products=set(items_df["parent_asin"]) #crea insieme con prodotti validi finali
qa_df = qa_df[
    qa_df["parent_asin"].isin(valid_products)
].copy() #filtriamo anche qa_df e teniamo solo le domande/rispose associate ai prodotti presenti in items_df

qa_df.to_parquet(QA_OUT, index=False) #riscriviamo file

print("Prodotti con QA dopo aver filtrato:", qa_df["parent_asin"].nunique())
print("Righe QA dopo filtro:", len(qa_df))

Prodotti finali con metadata + QA + rating: 22974
Prodotti con QA dopo aver filtrato: 22974
Righe QA dopo filtro: 180976


In [12]:
def parquet_count_rows(path: Path)-> int: #funzione che conta righe di un file parquet
    if not path.exists():
        return 0
    return pq.ParquetFile(path).metadata.num_rows 

summary={ #dizionario riassuntivo
    "items":parquet_count_rows(ITEMS_OUT),
    "qa":parquet_count_rows(QA_OUT),
    "ratings":parquet_count_rows(RATINGS_OUT),
}

summary_path=PROCESSED_DIR / "summary.json"
summary_path.write_text(json.dumps(summary,indent=2), encoding="utf-8")

summary

{'items': 22974, 'qa': 180976, 'ratings': 3484401}